# 05 — Single-Task Sentiment Control (`05_single_task_sentiment.ipynb`)

**Hypothesis Tested:** Does joint multi-task learning with the 6-class silver intent head help or hurt 4-class human sentiment classification compared to training the exact same `MLM+attn` encoder + attention pooling architecture on **sentiment alone**?

- **Architecture:**
  - Same 3-epoch in-domain MLM pretrained `xlm-roberta-base` encoder (`get_mlm_encoder(seed)` — reuses cached `mlm_seed{seed}` checkpoints when present).
  - Single-task 4-head attention pooler (`query_sent` over token states $\mathbf{H}$) + identical 2-layer GELU sentiment head (`Linear(768, 384) -> GELU -> Dropout(0.3) -> Linear(384, 4)`).
  - **No intent head, no intent loss.**
  - Early stopping (`PATIENCE = 3`) on validation **sentiment Macro-F1**.
- **Seeds:** `42, 43, 44` across both `runA` and `runB`.

## 1. Environment & Single Config Cell

In [ ]:
# !pip install -q transformers accelerate
# ── Colab / Repo-Root Setup ───────────────────────────────────────────────────
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("eval_utils.py"):
    if not os.path.exists("/content/Sinhala-joint-sentiment-and-intent"):
        subprocess.run(["git", "clone", "https://github.com/ThiwankaChanditha/Sinhala-joint-sentiment-and-intent.git", "/content/Sinhala-joint-sentiment-and-intent"], check=True)
    os.chdir("/content/Sinhala-joint-sentiment-and-intent")
elif not os.path.exists("eval_utils.py"):
    for _cand in ["..", "../..", "/content/Sinhala-joint-sentiment-and-intent"]:
        if os.path.exists(os.path.join(_cand, "eval_utils.py")):
            os.chdir(_cand)
            break
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import os
import gc
import math
import time
import json
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
)

from eval_utils import (
    SENT_CLASSES,
    NS,
    SEEDS,
    log_environment,
    set_all_seeds,
    load_splits,
    macro_f1_present,
    evaluate_and_save,
    summarize_Across_seeds,
)

warnings.filterwarnings("ignore")

# ── Single Config Cell (Identical Hyperparameters to Ablation) ────────────────
MODEL_NAME     = "xlm-roberta-base"
MAX_LEN        = 128
MLM_PROB       = 0.15
MLM_EPOCHS     = 3
MLM_BATCH      = 8
MLM_GRAD_ACC   = 2
MLM_LR         = 2e-5

FT_EPOCHS      = 10
FT_BATCH       = 8
FT_GRAD_ACC    = 2
FT_LR          = 2e-5
ENC_LR_MULT    = 0.1
WEIGHT_DECAY   = 0.01
DROPOUT        = 0.3
NUM_ATTN_HEADS = 4
PATIENCE       = 3

RUNS_TO_EXEC   = ["runA", "runB"]
NOTEBOOK_NAME  = "05_single_task_sentiment"
MODEL_TAG      = "single_task_sentiment"

WORK_DIR       = "/kaggle/working" if os.path.exists("/kaggle/working") else "."
SILVER_CSV     = None
GOLD_CSV       = None
RESULTS_DIR    = os.path.join(WORK_DIR, "results")
PREDS_DIR      = os.path.join(WORK_DIR, "preds")
ENC_DIR        = os.path.join(WORK_DIR, "mlm_encoders")
for d in (RESULTS_DIR, PREDS_DIR, ENC_DIR):
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env_info = log_environment()
print("Using device:", DEVICE)

## 2. Single-Task Sentiment Dataset & Architecture (`MLM + attn`, Sentiment Head Only)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_NAME)
mlm_collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=True, mlm_probability=MLM_PROB)

class SentDS(Dataset):
    def __init__(self, d):
        e = tok(d["text"].tolist(), truncation=True, max_length=MAX_LEN)
        self.ids, self.am = e["input_ids"], e["attention_mask"]
        self.s = d["sent_id"].tolist()
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, k):
        return {"input_ids": self.ids[k], "attention_mask": self.am[k], "sent": self.s[k]}

def collate_sent(batch):
    out = tok.pad([{"input_ids": b["input_ids"], "attention_mask": b["attention_mask"]} for b in batch], return_tensors="pt")
    out["sent"] = torch.tensor([b["sent"] for b in batch])
    return out

class SingleTaskAttnPooler(nn.Module):
    def __init__(self, hidden, heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(hidden, heads, dropout=dropout, batch_first=True)
        self.query_sent = nn.Parameter(torch.randn(1, 1, hidden))
        self.norm = nn.LayerNorm(hidden)

    def forward(self, H, mask):
        B, kpm = H.size(0), (mask == 0)
        o, _ = self.attn(self.query_sent.expand(B, -1, -1), H, H, key_padding_mask=kpm)
        return self.norm(o.squeeze(1))

class SingleTaskSentimentModel(nn.Module):
    def __init__(self, enc_path):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(enc_path)
        h = self.encoder.config.hidden_size
        self.pool = SingleTaskAttnPooler(h, NUM_ATTN_HEADS, DROPOUT)
        self.sent_fc = nn.Sequential(nn.Linear(h, h // 2), nn.GELU(), nn.Dropout(DROPOUT))
        self.sent_head = nn.Linear(h // 2, NS)

    def forward(self, input_ids, attention_mask):
        H = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        ps = self.pool(H, attention_mask)
        return self.sent_head(self.sent_fc(ps))

@torch.no_grad()
def predict_sent_probs(model, loader):
    model.eval()
    S = []
    for b in loader:
        s = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
        S.append(torch.softmax(s, -1).cpu())
    return torch.cat(S).numpy()

def class_weights(ids, K):
    c = np.bincount(ids, minlength=K).astype(float)
    return torch.tensor(np.where(c > 0, c.sum() / (K * np.maximum(c, 1)), 0.0), dtype=torch.float)

## 3. Domain MLM Stage (Cached) + Single-Task Fine-Tuning Loop

In [ ]:
def get_mlm_encoder(mlm_examples, run: str, seed: int) -> str:
    candidates = [
        os.path.join(ENC_DIR, f"mlm_{run}_seed{seed}"),
        os.path.join(ENC_DIR, f"mlm_seed{seed}"),
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, "config.json")):
            print(f"Using cached domain MLM encoder: {c}")
            return c

    out = candidates[0]
    set_all_seeds(seed)
    m = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).to(DEVICE)
    m.train()
    g = torch.Generator()
    g.manual_seed(seed)
    loader = DataLoader(mlm_examples, batch_size=MLM_BATCH, shuffle=True, collate_fn=mlm_collator, generator=g)
    opt = torch.optim.AdamW(m.parameters(), lr=MLM_LR, weight_decay=WEIGHT_DECAY)
    total = MLM_EPOCHS * math.ceil(len(loader) / MLM_GRAD_ACC)
    sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)
    for ep in range(MLM_EPOCHS):
        tl, n = 0.0, 0
        opt.zero_grad()
        for step, b in enumerate(loader):
            b = {k: v.to(DEVICE) for k, v in b.items()}
            if (b["labels"] != -100).any():
                loss = m(**b).loss
                (loss / MLM_GRAD_ACC).backward()
                tl += loss.item()
                n += 1
            if (step + 1) % MLM_GRAD_ACC == 0 or step + 1 == len(loader):
                nn.utils.clip_grad_norm_(m.parameters(), 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
        print(f"  [MLM {run} seed {seed}] epoch {ep+1}/{MLM_EPOCHS} loss {tl/max(n,1):.4f}")
    m.base_model.save_pretrained(out)
    del m, opt, sched
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return out

summaries = []

for run in RUNS_TO_EXEC:
    train_df, val_df, test_df = load_splits(run=run, silver_csv=SILVER_CSV, gold_csv=GOLD_CSV)
    mlm_texts = train_df["text"].tolist() + val_df["text"].tolist()
    assert not (set(mlm_texts) & set(test_df["text"])), "Leakage guard: test text in MLM corpus!"
    _e = tok(mlm_texts, truncation=True, max_length=MAX_LEN, return_special_tokens_mask=True)
    mlm_examples = [
        {"input_ids": a, "attention_mask": b, "special_tokens_mask": c}
        for a, b, c in zip(_e["input_ids"], _e["attention_mask"], _e["special_tokens_mask"])
    ]

    train_ds, val_ds, test_ds = SentDS(train_df), SentDS(val_df), SentDS(test_df)
    val_loader  = DataLoader(val_ds,  batch_size=FT_BATCH, shuffle=False, collate_fn=collate_sent)
    test_loader = DataLoader(test_ds, batch_size=FT_BATCH, shuffle=False, collate_fn=collate_sent)

    W_SENT = class_weights(train_df["sent_id"].values, NS).to(DEVICE)
    val_ys = val_df["sent_id"].values

    seed_payloads = []
    for seed in SEEDS:
        enc_path = get_mlm_encoder(mlm_examples, run=run, seed=seed)
        set_all_seeds(seed)
        model = SingleTaskSentimentModel(enc_path).to(DEVICE)
        g = torch.Generator()
        g.manual_seed(seed)
        loader = DataLoader(train_ds, batch_size=FT_BATCH, shuffle=True, collate_fn=collate_sent, generator=g)
        crit_s = nn.CrossEntropyLoss(weight=W_SENT)
        head_params = [p for n, p in model.named_parameters() if not n.startswith("encoder.")]
        opt = torch.optim.AdamW(
            [{"params": model.encoder.parameters(), "lr": FT_LR * ENC_LR_MULT},
             {"params": head_params, "lr": FT_LR}],
            weight_decay=WEIGHT_DECAY,
        )
        total = FT_EPOCHS * math.ceil(len(loader) / FT_GRAD_ACC)
        sched = get_linear_schedule_with_warmup(opt, max(1, total // 10), total)

        best_val, best_ep, best_state, bad = -1.0, 0, None, 0
        t0 = time.time()
        for ep in range(FT_EPOCHS):
            model.train()
            tl = 0.0
            opt.zero_grad()
            for step, b in enumerate(loader):
                s = model(b["input_ids"].to(DEVICE), b["attention_mask"].to(DEVICE))
                loss = crit_s(s, b["sent"].to(DEVICE))
                (loss / FT_GRAD_ACC).backward()
                tl += loss.item()
                if (step + 1) % FT_GRAD_ACC == 0 or step + 1 == len(loader):
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()
                    sched.step()
                    opt.zero_grad()
            vs = predict_sent_probs(model, val_loader)
            f_s = macro_f1_present(val_ys, vs.argmax(1), NS)
            print(f"  [SingleTask-Sent {run} s{seed}] ep {ep+1:>2} | train {tl/len(loader):.4f} | val Sent F1 {f_s:.3f}")
            if f_s > best_val:
                best_val, best_ep, bad = f_s, ep + 1, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= PATIENCE:
                    print("  early stop")
                    break

        model.load_state_dict(best_state)
        ts = predict_sent_probs(model, test_loader)
        del model, opt, sched, best_state
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        payload = evaluate_and_save(
            notebook_name=NOTEBOOK_NAME,
            model_name=MODEL_TAG,
            run=run,
            seed=seed,
            test_df=test_df,
            pred_sent_id=ts.argmax(1),
            pred_intent_id=None,
            sent_probs=ts,
            intent_probs=None,
            extra_meta={"best_epoch": best_ep, "best_val_sent_f1": best_val},
            results_dir=RESULTS_DIR,
            preds_dir=PREDS_DIR,
        )
        seed_payloads.append(payload)

    summaries.append(summarize_Across_seeds(seed_payloads))

display(pd.DataFrame(summaries)[["run", "model", "n_seeds", "sent_acc_str", "sent_f1_3cls_str", "sent_f1_4cls_str"]])